# RDFNet Kaggle Training (P100) with Periodic Validation/Test

This notebook trains RDFNet on your VOC foggy paired data, validates every 10 epochs, saves checkpoints every 5 epochs, and evaluates on VOC-fog and RTTS every 20 epochs.

It is wired for your Kaggle structure with VOC2012_filtered, VOC2012_paired, pairs.json, and the RTTS dataset mounted at /kaggle/input/datasets/tuncnguyn/rtts-dataset.

In [ ]:
import os
import re
import gc
import sys
import json
import glob
import shutil
import random
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

In [ ]:
# Install dependencies for Kaggle runtime
# Pin a CUDA 11.8 PyTorch stack that still supports P100 sm_60.
!pip -q install --upgrade --force-reinstall --no-cache-dir torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118
!pip -q install --upgrade --force-reinstall --no-cache-dir Pillow==10.4.0 tqdm numpy opencv-python pycocotools thop

In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))
    cap = torch.cuda.get_device_capability(0)
    if cap[0] < 7:
        print('WARNING: If training still fails on P100, restart the runtime after the torch reinstall cell and rerun from Cell 1.')

In [ ]:
# Clone RDFNet code into writable working directory
WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'RDFNet-main'

if not REPO_DIR.exists():
    !git clone https://github.com/PolarisFTL/RDFNet.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
print('Repo:', REPO_DIR)

In [ ]:
# -------------------------
# User-configurable settings
# -------------------------
# Paper-style baseline preset
BASELINE = {
    'total_epochs': 300,
    'test_every': 20,
    'val_every': 10,
    'save_every': 5,
    'batch_size': 16,        # If OOM on P100, set to 8
    'num_workers': 4,
    'input_shape': [640, 640],
    'optimizer_type': 'sgd',
    'init_lr': 1e-2,
    'momentum': 0.937,
    'weight_decay': 5e-4,
    'lr_decay_type': 'cos',
    'freeze_train': False,
    'freeze_epoch': 0,
    'fp16': False,
    'eval_confidence': 0.001,
    'eval_nms_iou': 0.5
}

TOTAL_EPOCHS = BASELINE['total_epochs']
TEST_EVERY = BASELINE['test_every']
VAL_EVERY = BASELINE['val_every']
SAVE_EVERY = BASELINE['save_every']
BATCH_SIZE = BASELINE['batch_size']
NUM_WORKERS = BASELINE['num_workers']
INPUT_SHAPE = BASELINE['input_shape']
FOG_LEVELS_FOR_TRAIN = ['high', 'mid', 'low']
VOC_FOG_EVAL_LEVEL = 'mid'

# Use your exact dataset mount first; fallback to auto-detection if not found.
MANUAL_DATASET_ROOT = Path('/kaggle/input/datasets/mdhabibourrahman/voc-2012-filtered/voc_2012')
MANUAL_RTTS_ROOT = MANUAL_DATASET_ROOT / 'processed' / 'VOC2012_filtered' / 'RTTS'


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate is not None and Path(candidate).exists():
            return Path(candidate)
    return None


def count_images_quick(root_dir, limit=30000):
    root_dir = Path(root_dir)
    if not root_dir.exists():
        return -1
    count = 0
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    for p in root_dir.rglob('*'):
        if p.is_file() and p.suffix.lower() in valid_ext and 'depth' not in p.name.lower():
            count += 1
            if count >= limit:
                break
    return count


def detect_voc_roots():
    if MANUAL_DATASET_ROOT.exists():
        filtered = find_first_existing([
            MANUAL_DATASET_ROOT / 'processed' / 'VOC2012_filtered',
            MANUAL_DATASET_ROOT / 'VOC2012_filtered'
        ])
        paired = find_first_existing([
            MANUAL_DATASET_ROOT / 'processed' / 'VOC2012_paired',
            MANUAL_DATASET_ROOT / 'VOC2012_paired'
        ])
        return MANUAL_DATASET_ROOT, filtered, paired

    input_root = Path('/kaggle/input')

    filtered_candidates = list(input_root.glob('**/processed/VOC2012_filtered'))
    filtered_candidates += list(input_root.glob('**/VOC2012_filtered'))
    filtered_candidates += list(input_root.glob('**/voc_2012_filtered'))

    paired_candidates = list(input_root.glob('**/processed/VOC2012_paired'))
    paired_candidates += list(input_root.glob('**/VOC2012_paired'))
    paired_candidates += list(input_root.glob('**/voc_2012_paired'))

    filtered = find_first_existing(filtered_candidates)
    paired = find_first_existing(paired_candidates)

    if filtered is None and paired is None:
        raise FileNotFoundError('Could not find VOC2012_filtered or VOC2012_paired under /kaggle/input.')

    root_candidate = filtered if filtered is not None else paired
    dataset_root = root_candidate.parent
    if dataset_root.name == 'processed':
        dataset_root = dataset_root.parent

    return dataset_root, filtered, paired


def detect_fog_base(dataset_root, paired_root, filtered_root):
    candidates = []

    if paired_root is not None:
        candidates.extend([
            paired_root / 'foggy',
            paired_root / 'Foggy',
            paired_root / 'fog',
            paired_root / 'Fog',
            paired_root / 'hazy',
            paired_root / 'Hazy'
        ])

    candidates.extend([
        dataset_root / 'processed' / 'VOC2012_foggy',
        dataset_root / 'processed' / 'VOC2012_fog',
        dataset_root / 'VOC2012_foggy',
        dataset_root / 'VOC2012_fog',
        dataset_root / 'foggy',
        dataset_root / 'fog',
        dataset_root / 'Foggy',
        dataset_root / 'Fog'
    ])

    if filtered_root is not None:
        candidates.extend([
            filtered_root / 'foggy',
            filtered_root / 'Foggy',
            filtered_root / 'fog',
            filtered_root / 'Fog'
        ])

    # Discover additional fog-like directories from the known dataset root.
    for pattern in ['**/*fog*', '**/*Fog*', '**/*hazy*', '**/*Hazy*']:
        for p in dataset_root.glob(pattern):
            if p.is_dir():
                candidates.append(p)

    # Deduplicate while preserving order.
    uniq = []
    seen = set()
    for c in candidates:
        c = Path(c)
        if c in seen:
            continue
        seen.add(c)
        if c.exists():
            uniq.append(c)

    scored = []
    for c in uniq:
        scored.append((count_images_quick(c), c))

    scored.sort(key=lambda x: x[0], reverse=True)
    if len(scored) > 0:
        print('Fog base candidates (image_count, path):')
        for n, p in scored[:15]:
            print(' ', n, p)

    for n, p in scored:
        if n > 0:
            return p

    return find_first_existing(uniq)


DATASET_ROOT, VOC2012_FILTERED, VOC2012_PAIRED = detect_voc_roots()

if VOC2012_FILTERED is None:
    VOC2012_FILTERED = find_first_existing([
        DATASET_ROOT / 'processed' / 'VOC2012_filtered',
        DATASET_ROOT / 'VOC2012_filtered',
        DATASET_ROOT / 'voc_2012_filtered'
    ])
if VOC2012_PAIRED is None:
    VOC2012_PAIRED = find_first_existing([
        DATASET_ROOT / 'processed' / 'VOC2012_paired',
        DATASET_ROOT / 'VOC2012_paired',
        DATASET_ROOT / 'voc_2012_paired'
    ])

VOC2012_FILTERED_IMAGESETS = VOC2012_FILTERED / 'ImageSets' / 'Main'
VOC2012_FILTERED_ANN = VOC2012_FILTERED / 'Annotations'
VOC2012_FILTERED_IMG = VOC2012_FILTERED / 'JPEGImages'
VOC2012_PAIRED_IMAGESETS = VOC2012_PAIRED / 'ImageSets' / 'Main'
VOC2012_PAIRED_CLEAN_ANN = find_first_existing([
    VOC2012_PAIRED / 'clean' / 'Annotations',
    VOC2012_PAIRED / 'Annotations'
])
VOC2012_PAIRED_CLEAN_IMG = find_first_existing([
    VOC2012_PAIRED / 'clean' / 'JPEGImages',
    VOC2012_PAIRED / 'JPEGImages'
])
VOC2012_PAIRED_FOG_BASE = detect_fog_base(DATASET_ROOT, VOC2012_PAIRED, VOC2012_FILTERED)
PAIR_JSON = find_first_existing([
    VOC2012_PAIRED / 'pairs.json',
    DATASET_ROOT / 'pairs.json'
])

# RTTS can be either VOC2007 format or plain RTTS/{Annotations,JPEGImages}
RTTS_ROOT = find_first_existing([
    MANUAL_RTTS_ROOT,
    DATASET_ROOT / 'RTTS',
    DATASET_ROOT / 'processed' / 'RTTS',
    Path('/kaggle/input/datasets/tuncnguyn/rtts-dataset')
])

print('DATASET_ROOT:', DATASET_ROOT)
print('VOC2012_FILTERED:', VOC2012_FILTERED)
print('VOC2012_PAIRED:', VOC2012_PAIRED)
print('VOC2012_PAIRED_FOG_BASE:', VOC2012_PAIRED_FOG_BASE)
print('PAIR_JSON:', PAIR_JSON)
print('RTTS_ROOT:', RTTS_ROOT)
print('BASELINE PRESET:', BASELINE)

In [ ]:
# Print dataset folder structure (run this before data list generation)

def print_tree(root, max_depth=3, max_entries=200):
    root = Path(root)
    if not root.exists():
        print(f'Not found: {root}')
        return

    print(f'\n===== TREE: {root} =====')
    shown = 0
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        if shown >= max_entries:
            print('... truncated ...')
            break
        prefix = '  ' * (depth - 1)
        marker = '/' if path.is_dir() else ''
        print(f'{prefix}{rel}{marker}')
        shown += 1


def safe_count(glob_path):
    try:
        return len(list(glob_path))
    except Exception:
        return -1


print_tree(DATASET_ROOT, max_depth=3, max_entries=250)
print_tree(VOC2012_FILTERED, max_depth=3, max_entries=250)
print_tree(VOC2012_PAIRED, max_depth=4, max_entries=350)
if VOC2012_PAIRED_FOG_BASE is not None:
    print_tree(VOC2012_PAIRED_FOG_BASE, max_depth=3, max_entries=350)
if RTTS_ROOT is not None:
    print_tree(RTTS_ROOT, max_depth=3, max_entries=250)

print('\n===== QUICK COUNTS =====')
print('Filtered XML:', safe_count(VOC2012_FILTERED_ANN.glob('*.xml')))
print('Filtered JPG:', safe_count(VOC2012_FILTERED_IMG.glob('*.jpg')))
if VOC2012_PAIRED_CLEAN_ANN is not None:
    print('Paired Clean XML:', safe_count(VOC2012_PAIRED_CLEAN_ANN.glob('*.xml')))
if VOC2012_PAIRED_CLEAN_IMG is not None:
    print('Paired Clean JPG:', safe_count(VOC2012_PAIRED_CLEAN_IMG.glob('*.jpg')))
if VOC2012_PAIRED_FOG_BASE is not None:
    for lvl in ['high', 'mid', 'low']:
        cand1 = VOC2012_PAIRED_FOG_BASE / lvl / 'JPEGImages'
        cand2 = VOC2012_PAIRED_FOG_BASE / lvl
        c1 = safe_count(cand1.glob('*')) if cand1.exists() else -1
        c2 = safe_count(cand2.glob('*')) if cand2.exists() else -1
        print(f'Fog level {lvl}: JPEGImages={c1}, level_root={c2}')

In [ ]:
# Clean previous training outputs before starting a new run
LOGS_DIR = REPO_DIR / 'logs'
if LOGS_DIR.exists():
    shutil.rmtree(LOGS_DIR)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
print('Cleared previous logs in', LOGS_DIR)

In [ ]:
# Build training/validation text files expected by this repository

PAIRED_ANN_ROOT = VOC2012_PAIRED_CLEAN_ANN if VOC2012_PAIRED_CLEAN_ANN is not None else VOC2012_FILTERED_ANN
PAIRED_IMG_ROOT = VOC2012_PAIRED_CLEAN_IMG if VOC2012_PAIRED_CLEAN_IMG is not None else VOC2012_FILTERED_IMG
FOG_BASE = VOC2012_PAIRED_FOG_BASE


def find_image_with_ext(root_dir, image_id):
    if root_dir is None:
        return None
    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        p = root_dir / f'{image_id}{ext}'
        if p.exists():
            return p
    return None


def is_valid_rgb_image(path_obj):
    name = path_obj.name.lower()
    if path_obj.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp'}:
        return False
    if 'depth' in name:
        return False
    return True


def get_level_img_dir(level_name):
    if FOG_BASE is None:
        return None
    candidates = [
        FOG_BASE / level_name / 'JPEGImages',
        FOG_BASE / level_name / 'images',
        FOG_BASE / level_name,
        FOG_BASE / 'VOC2012_foggy' / level_name / 'JPEGImages',
        FOG_BASE / 'VOC2012_foggy' / level_name
    ]
    return find_first_existing(candidates)


def load_class_list():
    classes_file_candidates = [
        VOC2012_FILTERED / 'classes.txt',
        VOC2012_FILTERED / 'processed' / 'classes.txt',
        DATASET_ROOT / 'classes.txt',
        REPO_DIR / 'model_data' / 'rtts_classes.txt'
    ]
    classes_file = find_first_existing(classes_file_candidates)
    if classes_file is None:
        raise FileNotFoundError('Could not find classes.txt for the VOC filtered dataset.')
    with open(classes_file, 'r', encoding='utf-8') as f:
        classes = [x.strip() for x in f if x.strip()]
    return classes, classes_file


def load_class_mapping():
    mapping_file_candidates = [
        VOC2012_FILTERED / 'class_mapping.txt',
        DATASET_ROOT / 'class_mapping.txt'
    ]
    mapping_file = find_first_existing(mapping_file_candidates)
    mapping = {}
    if mapping_file is None:
        return mapping

    with open(mapping_file, 'r', encoding='utf-8') as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith('#'):
                continue
            if ',' in s:
                parts = [p.strip() for p in s.split(',')]
            elif '->' in s:
                parts = [p.strip() for p in s.split('->')]
            else:
                parts = s.split()
            if len(parts) >= 2:
                mapping[parts[0]] = parts[1]
    return mapping


def parse_xml_boxes(xml_path, class_to_idx, class_mapping):
    if not xml_path.exists():
        return []
    root = ET.parse(xml_path).getroot()
    out = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        name = class_mapping.get(name, name)
        difficult = obj.find('difficult')
        if difficult is not None and difficult.text.strip() == '1':
            continue
        if name not in class_to_idx:
            continue
        box = obj.find('bndbox')
        xmin = int(float(box.find('xmin').text))
        ymin = int(float(box.find('ymin').text))
        xmax = int(float(box.find('xmax').text))
        ymax = int(float(box.find('ymax').text))
        out.append((xmin, ymin, xmax, ymax, class_to_idx[name]))
    return out


def make_line(image_path, boxes):
    if len(boxes) == 0:
        return None
    box_str = ' '.join([','.join(map(str, b)) for b in boxes])
    return f'{image_path} {box_str}'


def collect_level_image_map(level_name):
    img_dir = get_level_img_dir(level_name)
    if img_dir is None or (not img_dir.exists()):
        print(f'Level {level_name}: image dir not found')
        return {}
    image_map = {}
    for p in img_dir.rglob('*'):
        if p.is_file() and is_valid_rgb_image(p):
            image_map[p.stem] = p
    print(f'Level {level_name}: discovered images={len(image_map)} from {img_dir}')
    return image_map


def build_level_pairs_from_images(level_name):
    image_map = collect_level_image_map(level_name)
    stats = {'total_images': 0, 'missing_xml': 0, 'empty_boxes': 0, 'missing_clear': 0, 'usable': 0}
    pairs = []

    for image_id, fog_img_path in image_map.items():
        stats['total_images'] += 1

        xml_path = PAIRED_ANN_ROOT / f'{image_id}.xml'
        if not xml_path.exists():
            stats['missing_xml'] += 1
            continue

        boxes = parse_xml_boxes(xml_path, class_to_idx, class_mapping)
        if len(boxes) == 0:
            stats['empty_boxes'] += 1
            continue

        clear_path = find_image_with_ext(PAIRED_IMG_ROOT, image_id)
        if clear_path is None:
            clear_path = find_image_with_ext(VOC2012_FILTERED_IMG, image_id)
        if clear_path is None:
            stats['missing_clear'] += 1
            continue

        line = make_line(str(fog_img_path.resolve()), boxes)
        if line is not None:
            pairs.append((line, str(clear_path.resolve())))
            stats['usable'] += 1

    print(f"Level {level_name}: total={stats['total_images']} missing_xml={stats['missing_xml']} empty_boxes={stats['empty_boxes']} missing_clear={stats['missing_clear']} usable={stats['usable']}")
    return pairs


def collect_all_fog_images_fallback():
    if FOG_BASE is None or (not FOG_BASE.exists()):
        return {}
    image_map = {}
    for p in FOG_BASE.rglob('*'):
        if p.is_file() and is_valid_rgb_image(p):
            image_map[p.stem] = p
    print(f'Fallback image discovery under fog base: {len(image_map)} images')
    return image_map


classes, classes_file = load_class_list()
class_mapping = load_class_mapping()
class_to_idx = {c: i for i, c in enumerate(classes)}

print('Classes:', classes)
print('Using classes file:', classes_file)
print('Annotation root:', PAIRED_ANN_ROOT)
print('Image root:', PAIRED_IMG_ROOT)
print('Fog base:', FOG_BASE)

if PAIRED_ANN_ROOT is None or not PAIRED_ANN_ROOT.exists():
    raise FileNotFoundError('Could not find paired annotation root.')
if PAIRED_IMG_ROOT is None or not PAIRED_IMG_ROOT.exists():
    raise FileNotFoundError('Could not find paired clean image root.')
if FOG_BASE is None or not FOG_BASE.exists():
    raise FileNotFoundError('Could not find foggy dataset root.')

all_level_pairs = []
for level in FOG_LEVELS_FOR_TRAIN:
    all_level_pairs.extend(build_level_pairs_from_images(level))

if len(all_level_pairs) == 0:
    # Fallback: ignore level folders and use all fog images found recursively.
    fallback_map = collect_all_fog_images_fallback()
    for image_id, fog_img_path in fallback_map.items():
        xml_path = PAIRED_ANN_ROOT / f'{image_id}.xml'
        if not xml_path.exists():
            continue
        boxes = parse_xml_boxes(xml_path, class_to_idx, class_mapping)
        if len(boxes) == 0:
            continue
        clear_path = find_image_with_ext(PAIRED_IMG_ROOT, image_id)
        if clear_path is None:
            clear_path = find_image_with_ext(VOC2012_FILTERED_IMG, image_id)
        if clear_path is None:
            continue
        line = make_line(str(fog_img_path.resolve()), boxes)
        if line is not None:
            all_level_pairs.append((line, str(clear_path.resolve())))

if len(all_level_pairs) == 0:
    raise ValueError('No usable pairs found from fog images. Check fog image paths, annotation names, and class labels.')

# Shuffle and split 90/10 for train/val
random.seed(42)
random.shuffle(all_level_pairs)
split_idx = max(1, int(len(all_level_pairs) * 0.9))
train_pairs = all_level_pairs[:split_idx]
val_pairs_global = all_level_pairs[split_idx:]

train_fog_lines = [x[0] for x in train_pairs]
train_clear_lines = [x[1] for x in train_pairs]

# Prefer val from selected eval level if possible
eval_level_pairs = build_level_pairs_from_images(VOC_FOG_EVAL_LEVEL)
if len(eval_level_pairs) > 0:
    val_fog_lines = [x[0] for x in eval_level_pairs]
else:
    val_fog_lines = [x[0] for x in val_pairs_global]

(REPO_DIR / '2007_train_fog.txt').write_text('\n'.join(train_fog_lines), encoding='utf-8')
(REPO_DIR / '2007_train.txt').write_text('\n'.join(train_clear_lines), encoding='utf-8')
(REPO_DIR / '2007_val_fog.txt').write_text('\n'.join(val_fog_lines), encoding='utf-8')

if len(train_fog_lines) == 0 or len(val_fog_lines) == 0:
    raise ValueError(f'No usable training/validation samples were built. train={len(train_fog_lines)}, val={len(val_fog_lines)}')

classes_kaggle = REPO_DIR / 'model_data' / 'classes_kaggle.txt'
classes_kaggle.write_text('\n'.join(classes) + '\n', encoding='utf-8')

print('Train fog samples:', len(train_fog_lines))
print('Train clear samples:', len(train_clear_lines))
print('Val fog samples:', len(val_fog_lines))
print('Classes file for config:', classes_kaggle)

In [ ]:
# Patch config.py values for this training schedule

config_path = REPO_DIR / 'config.py'


def replace_assign(text, key, value_literal):
    pattern = rf'^{key}\s*=\s*.*$'
    repl = f'{key:<18}= {value_literal}'
    return re.sub(pattern, repl, text, flags=re.MULTILINE)


def patch_config(init_epoch, end_epoch, model_path_literal):
    txt = config_path.read_text(encoding='utf-8')

    # Core runtime
    txt = replace_assign(txt, 'Cuda', 'True')
    txt = replace_assign(txt, 'distributed', 'False')
    txt = replace_assign(txt, 'sync_bn', 'False')
    txt = replace_assign(txt, 'fp16', str(BASELINE['fp16']))

    # Model/data
    txt = replace_assign(txt, 'classes_path', repr(str((REPO_DIR / 'model_data' / 'classes_kaggle.txt').resolve())))
    txt = replace_assign(txt, 'anchors_path', repr('model_data/yolo_anchors.txt'))
    txt = replace_assign(txt, 'model_path', repr(model_path_literal))
    txt = replace_assign(txt, 'input_shape', repr(INPUT_SHAPE))

    # Epoch and batch schedule
    txt = replace_assign(txt, 'Init_Epoch', str(init_epoch))
    txt = replace_assign(txt, 'Freeze_Epoch', str(BASELINE['freeze_epoch']))
    txt = replace_assign(txt, 'Freeze_batch_size', str(BATCH_SIZE))
    txt = replace_assign(txt, 'UnFreeze_Epoch', str(end_epoch))
    txt = replace_assign(txt, 'Unfreeze_batch_size', str(BATCH_SIZE))
    txt = replace_assign(txt, 'Freeze_Train', str(BASELINE['freeze_train']))

    # Optimization
    txt = replace_assign(txt, 'Init_lr', str(BASELINE['init_lr']))
    txt = replace_assign(txt, 'Min_lr', 'Init_lr * 0.01')
    txt = replace_assign(txt, 'optimizer_type', repr(BASELINE['optimizer_type']))
    txt = replace_assign(txt, 'momentum', str(BASELINE['momentum']))
    txt = replace_assign(txt, 'weight_decay', str(BASELINE['weight_decay']))
    txt = replace_assign(txt, 'lr_decay_type', repr(BASELINE['lr_decay_type']))

    # Logging/eval cadence
    txt = replace_assign(txt, 'save_period', str(SAVE_EVERY))
    txt = replace_assign(txt, 'save_dir', repr('logs'))
    txt = replace_assign(txt, 'eval_flag', 'True')
    txt = replace_assign(txt, 'eval_period', str(VAL_EVERY))
    txt = replace_assign(txt, 'num_workers', str(NUM_WORKERS))

    # Annotation files produced by the notebook
    txt = replace_assign(txt, 'train_annotation_path', repr('2007_train_fog.txt'))
    txt = replace_assign(txt, 'val_annotation_path', repr('2007_val_fog.txt'))
    txt = replace_assign(txt, 'clear_annotation_path', repr('2007_train.txt'))

    config_path.write_text(txt, encoding='utf-8')


# initial patch preview
patch_config(0, TEST_EVERY, 'model_data/yolov7_tiny_weights.pth')
print('config.py patched with baseline settings')

In [ ]:
# Evaluation helpers for VOC-fog and RTTS

from PIL import Image
from yolo import YOLO
from utils.utils_map import get_map


def read_ids_or_all(ann_dir, ids_file=None):
    if ids_file is not None and Path(ids_file).exists():
        return [x.strip() for x in Path(ids_file).read_text().splitlines() if x.strip()]
    return sorted([p.stem for p in Path(ann_dir).glob('*.xml')])


def create_gt_file(xml_path, out_txt, class_set):
    root = ET.parse(xml_path).getroot()
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        name = class_mapping.get(name, name)
        if name not in class_set:
            continue
        box = obj.find('bndbox')
        left = int(float(box.find('xmin').text))
        top = int(float(box.find('ymin').text))
        right = int(float(box.find('xmax').text))
        bottom = int(float(box.find('ymax').text))
        lines.append(f'{name} {left} {top} {right} {bottom}')
    Path(out_txt).write_text('\n'.join(lines), encoding='utf-8')


def evaluate_dataset(model_path, classes_path, ann_dir, img_dir, ids_file=None, tag='eval'):
    ann_dir = Path(ann_dir)
    img_dir = Path(img_dir)
    ids = read_ids_or_all(ann_dir, ids_file)

    out_dir = REPO_DIR / f'map_out_{tag}'
    if out_dir.exists():
        shutil.rmtree(out_dir)
    (out_dir / 'ground-truth').mkdir(parents=True, exist_ok=True)
    (out_dir / 'detection-results').mkdir(parents=True, exist_ok=True)

    with open(classes_path, 'r', encoding='utf-8') as f:
        class_names = [x.strip() for x in f if x.strip()]
    class_set = set(class_names)

    yolo = YOLO(
        model_path=str(model_path),
        classes_path=str(classes_path),
        anchors_path=str((REPO_DIR / 'model_data' / 'yolo_anchors.txt').resolve()),
        confidence=BASELINE['eval_confidence'],
        nms_iou=BASELINE['eval_nms_iou']
    )

    valid_count = 0
    for img_id in tqdm(ids, desc=f'{tag}: infer'):
        image_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            p = img_dir / f'{img_id}{ext}'
            if p.exists():
                image_path = p
                break
        xml_path = ann_dir / f'{img_id}.xml'
        if image_path is None or (not xml_path.exists()):
            continue
        valid_count += 1
        image = Image.open(image_path)
        yolo.get_map_txt(img_id, image, class_names, str(out_dir))
        create_gt_file(xml_path, out_dir / 'ground-truth' / f'{img_id}.txt', class_set)

    if valid_count == 0:
        print(f'[{tag}] No valid images found for evaluation.')
        return 0.0

    m = get_map(0.5, False, score_threhold=0.5, path=str(out_dir))
    print(f'[{tag}] mAP@0.5 = {m:.4f}')
    return float(m)


def get_level_paths(level_name):
    level_name = str(level_name).strip().lower()
    fog_base = Path(FOG_BASE) if FOG_BASE is not None else None

    if fog_base is None or (not fog_base.exists()):
        raise FileNotFoundError('FOG_BASE is missing. Re-run the settings and data preparation cells first.')

    level_aliases = {
        'high': ['high', 'dense', 'heavy'],
        'mid': ['mid', 'medium', 'moderate'],
        'low': ['low', 'light']
    }
    names = level_aliases.get(level_name, [level_name])

    def has_images(path_obj):
        if path_obj is None or (not Path(path_obj).exists()):
            return False
        p = Path(path_obj)
        return any(p.rglob('*.jpg')) or any(p.rglob('*.jpeg')) or any(p.rglob('*.png')) or any(p.rglob('*.bmp'))

    def scan_level_dir(root, level_tokens):
        for p in root.rglob('*'):
            if not p.is_dir():
                continue
            pname = p.name.lower()
            if not any(tok in pname for tok in level_tokens):
                continue
            ann = find_first_existing([
                p / 'Annotations',
                p / 'annotations',
                p / 'XML',
                p / 'xml',
                VOC2012_PAIRED_CLEAN_ANN,
                VOC2012_FILTERED_ANN
            ])
            img = find_first_existing([
                p / 'JPEGImages',
                p / 'jpegimages',
                p / 'images',
                p,
                VOC2012_PAIRED_CLEAN_IMG,
                VOC2012_FILTERED_IMG
            ])
            if ann is not None and img is not None and Path(ann).exists() and has_images(img):
                return Path(ann), Path(img)
        return None, None

    structured_candidates = []
    for n in names:
        structured_candidates.extend([
            (fog_base / n / 'Annotations', fog_base / n / 'JPEGImages'),
            (fog_base / n / 'annotations', fog_base / n / 'images'),
            (VOC2012_PAIRED_CLEAN_ANN, fog_base / n / 'JPEGImages'),
            (VOC2012_PAIRED_CLEAN_ANN, fog_base / n)
        ])

    for ann_c, img_c in structured_candidates:
        ann = find_first_existing([ann_c, VOC2012_PAIRED_CLEAN_ANN, VOC2012_FILTERED_ANN])
        img = find_first_existing([img_c])
        if ann is not None and img is not None and Path(ann).exists() and has_images(img):
            return Path(ann), Path(img)

    ann, img = scan_level_dir(fog_base, names)
    if ann is not None and img is not None:
        return ann, img

    ann = find_first_existing([VOC2012_PAIRED_CLEAN_ANN, VOC2012_FILTERED_ANN])
    img = fog_base
    if ann is None or img is None:
        raise FileNotFoundError(f'Could not resolve annotation/image paths for level "{level_name}" under {fog_base}')
    return Path(ann), Path(img)


# Safety net: if user is running an older cell snapshot in Kaggle, define a minimal resolver.
if 'get_level_paths' not in globals():
    def get_level_paths(level_name):
        ann = find_first_existing([VOC2012_PAIRED_CLEAN_ANN, VOC2012_FILTERED_ANN])
        img = Path(FOG_BASE) if FOG_BASE is not None else None
        if ann is None or img is None or (not Path(img).exists()):
            raise FileNotFoundError('Could not resolve VOC fog evaluation paths.')
        return Path(ann), Path(img)


# Use fog-level annotation/image folders directly for VOC-fog evaluation
voc_eval_ann_dir, voc_eval_img_dir = get_level_paths(VOC_FOG_EVAL_LEVEL)
voc_eval_ids_file = None

# RTTS supports both formats:
# 1) RTTS/VOC2007/{Annotations,JPEGImages,ImageSets/Main/test.txt}
# 2) RTTS/{Annotations,JPEGImages}
rtts_eval_ann_dir = None
rtts_eval_img_dir = None
rtts_eval_ids_file = None

if RTTS_ROOT is not None and Path(RTTS_ROOT).exists():
    voc2007_ann = Path(RTTS_ROOT) / 'VOC2007' / 'Annotations'
    voc2007_img = Path(RTTS_ROOT) / 'VOC2007' / 'JPEGImages'
    flat_ann = Path(RTTS_ROOT) / 'Annotations'
    flat_img = Path(RTTS_ROOT) / 'JPEGImages'

    if voc2007_ann.exists() and voc2007_img.exists():
        rtts_eval_ann_dir = voc2007_ann
        rtts_eval_img_dir = voc2007_img
        rtts_eval_ids_file = find_first_existing([
            Path(RTTS_ROOT) / 'VOC2007' / 'ImageSets' / 'Main' / 'test.txt',
            Path(RTTS_ROOT) / 'VOC2007' / 'ImageSets' / 'Main' / 'val.txt'
        ])
    elif flat_ann.exists() and flat_img.exists():
        rtts_eval_ann_dir = flat_ann
        rtts_eval_img_dir = flat_img
        rtts_eval_ids_file = None

print('VOC eval ann:', voc_eval_ann_dir)
print('VOC eval img:', voc_eval_img_dir)
print('RTTS eval ann:', rtts_eval_ann_dir)
print('RTTS eval img:', rtts_eval_img_dir)

In [ ]:
# Train in 20-epoch chunks and test after each chunk

metrics = []
prev_model_path = 'model_data/yolov7_tiny_weights.pth'
python_executable = sys.executable

if len(train_fog_lines) == 0 or len(val_fog_lines) == 0:
    raise ValueError('Dataset setup produced zero training or validation samples. Fix the dataset paths before training.')

for end_epoch in range(TEST_EVERY, TOTAL_EPOCHS + 1, TEST_EVERY):
    start_epoch = end_epoch - TEST_EVERY
    print('=' * 90)
    print(f'Training chunk: {start_epoch} -> {end_epoch}')

    patch_config(start_epoch, end_epoch, prev_model_path)

    cmd = [python_executable, 'train.py']
    result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError('Training failed for chunk ending at epoch ' + str(end_epoch))

    ckpt_matches = sorted((REPO_DIR / 'logs').glob(f'ep{end_epoch:03d}-*.pth'))
    if len(ckpt_matches) == 0:
        raise FileNotFoundError(f'Checkpoint for epoch {end_epoch} not found in logs/')
    current_ckpt = ckpt_matches[-1]
    prev_model_path = str(current_ckpt.resolve())
    print('Checkpoint:', current_ckpt)

    voc_map = evaluate_dataset(
        model_path=current_ckpt,
        classes_path=REPO_DIR / 'model_data' / 'classes_kaggle.txt',
        ann_dir=voc_eval_ann_dir,
        img_dir=voc_eval_img_dir,
        ids_file=voc_eval_ids_file,
        tag=f'vocfog_e{end_epoch}'
    )

    if rtts_eval_ann_dir is not None and Path(rtts_eval_ann_dir).exists():
        rtts_map = evaluate_dataset(
            model_path=current_ckpt,
            classes_path=REPO_DIR / 'model_data' / 'classes_kaggle.txt',
            ann_dir=rtts_eval_ann_dir,
            img_dir=rtts_eval_img_dir,
            ids_file=rtts_eval_ids_file,
            tag=f'rtts_e{end_epoch}'
        )
    else:
        rtts_map = None
        print('RTTS path not found, skipping RTTS evaluation for this chunk.')

    metrics.append({
        'epoch': end_epoch,
        'checkpoint': str(current_ckpt),
        'voc_fog_map50': voc_map,
        'rtts_map50': rtts_map
    })

    gc.collect()

print('All chunks complete.')

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame(metrics)
display(metrics_df)

metrics_out = REPO_DIR / 'logs' / 'kaggle_periodic_eval_metrics.csv'
metrics_df.to_csv(metrics_out, index=False)
print('Saved:', metrics_out)